In [ ]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

from train_identity_risk_catboost import (
    CATEGORICAL_FEATURES,
    DATA_PATH,
    MODEL_DIR,
    MODEL_PATH,
    RANDOM_SEED,
    TARGET,
    build_feature_columns,
    load_training_data,
)


In [ ]:
df = load_training_data(DATA_PATH)
print(f"Data successfully loaded. Total rows: {len(df)}")
df.head()

In [ ]:
df = df.dropna().reset_index(drop=True)
df.shape

In [ ]:
feature_cols = build_feature_columns(df)
categorical_features = [column for column in CATEGORICAL_FEATURES if column in feature_cols]

print(f"Target: {TARGET}")
print(f"Feature count: {len(feature_cols)}")
print(f"Categorical feature count: {len(categorical_features)}")
feature_cols

In [ ]:
X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y,
)

train_pool = Pool(X_train, y_train, cat_features=categorical_features)
test_pool = Pool(X_test, y_test, cat_features=categorical_features)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
model = CatBoostClassifier(
    task_type="GPU",
    auto_class_weights="Balanced",
    use_best_model=True,
    iterations=10000,
    depth=6,
    learning_rate=0.01,
    random_seed=RANDOM_SEED,
    verbose=100,
    loss_function="Logloss",
    eval_metric="AUC",
)

print("CatBoost model initialized with GPU acceleration and class weighting.")

In [ ]:
print("Starting training...")
model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=500)
print("Training complete. Best iteration selected.")

In [ ]:
y_pred_prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_prob)
print(f"Final Model AUC on unseen Test Data: {auc:.4f}")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_model(str(MODEL_PATH))
print(f"Model saved as {MODEL_PATH}")

In [ ]:
y_train_pred_prob = model.predict_proba(X_train)[:, 1]
train_auc = roc_auc_score(y_train, y_train_pred_prob)

print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC: {auc:.4f}")

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print(f"Train Accuracy: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")